<a href="https://colab.research.google.com/github/Mgonzalez980/escalonamento-de-processos/blob/main/Atividade_7_Agendamento_Processos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Notebook — Desafio 7

**Agendamento de processos: FCFS, Round Robin e SJF**

Atividade baseada no notebook de aula do professor. Foram mantidos os processos originais e adicionado P4 (chegada 3, burst 1).

In [ ]:
from collections import deque
from dataclasses import dataclass
import matplotlib.pyplot as plt

@dataclass
class Processo:
    nome: str
    chegada: int
    burst: int

processos = [
    Processo('P1', 0, 6),
    Processo('P2', 1, 2),
    Processo('P3', 2, 4),
    Processo('P4', 3, 1),
]

processos

## 1. Funções de apoio

As fatias são representadas por `(início, fim, processo)`. As métricas são resposta, espera e retorno.

In [ ]:
CORES = {'P1':'#1677c8', 'P2':'#11a6a0', 'P3':'#f59f00', 'P4':'#7c3aed'}

def gantt(linha, titulo):
    fig, ax = plt.subplots(figsize=(12, 2.2))
    for inicio, fim, nome in linha:
        ax.barh(0, fim-inicio, left=inicio, color=CORES.get(nome, '#64748b'), edgecolor='white', height=.55)
        ax.text((inicio+fim)/2, 0, nome, ha='center', va='center', color='white', weight='bold')
        ax.text(inicio, -.38, str(inicio), ha='center')
    ax.text(linha[-1][1], -.38, str(linha[-1][1]), ha='center')
    ax.set_yticks([])
    ax.set_xlabel('tempo (ms)')
    ax.set_title(titulo)
    ax.spines[['top','right','left']].set_visible(False)
    plt.show()

def metricas(linha, processos):
    dados = []
    for p in processos:
        fatias = [(i, f) for i, f, nome in linha if nome == p.nome]
        inicio, fim = fatias[0][0], fatias[-1][1]
        resposta = inicio - p.chegada
        retorno = fim - p.chegada
        espera = retorno - p.burst
        dados.append((p.nome, resposta, espera, retorno))
    return dados

def mostrar_metricas(linha, processos):
    dados = metricas(linha, processos)
    print('Processo | Resposta | Espera | Retorno')
    for d in dados:
        print(f'{d[0]:>8} | {d[1]:>8} | {d[2]:>6} | {d[3]:>7}')
    medias = [sum(x[i] for x in dados)/len(dados) for i in (1,2,3)]
    print(f'Médias   | {medias[0]:8.2f} | {medias[1]:6.2f} | {medias[2]:7.2f}')

## 2. FCFS

Executa os processos por ordem de chegada, sem interromper o processo atual.

In [ ]:
def fcfs(processos):
    tempo = 0
    linha = []
    for p in sorted(processos, key=lambda p: p.chegada):
        inicio = max(tempo, p.chegada)
        fim = inicio + p.burst
        linha.append((inicio, fim, p.nome))
        tempo = fim
    return linha

linha_fcfs = fcfs(processos)
gantt(linha_fcfs, 'FCFS')
mostrar_metricas(linha_fcfs, processos)

## 3. Round Robin

Cada processo executa por até um quantum e, caso não termine, retorna ao fim da fila. Processos que chegam durante a fatia entram na fila antes do processo interrompido.

In [ ]:
def round_robin(processos, quantum):
    if quantum <= 0:
        raise ValueError('O quantum deve ser positivo.')
    ordenados = sorted(enumerate(processos), key=lambda item: (item[1].chegada, item[0]))
    restantes = {p.nome: p.burst for p in processos}
    fila = deque()
    linha = []
    tempo = 0
    proximo = 0
    concluidos = 0
    while concluidos < len(processos):
        if not fila and proximo < len(ordenados) and tempo < ordenados[proximo][1].chegada:
            tempo = ordenados[proximo][1].chegada
        while proximo < len(ordenados) and ordenados[proximo][1].chegada <= tempo:
            fila.append(ordenados[proximo][1])
            proximo += 1
        if not fila:
            continue
        atual = fila.popleft()
        duracao = min(quantum, restantes[atual.nome])
        linha.append((tempo, tempo + duracao, atual.nome))
        tempo += duracao
        restantes[atual.nome] -= duracao
        while proximo < len(ordenados) and ordenados[proximo][1].chegada <= tempo:
            fila.append(ordenados[proximo][1])
            proximo += 1
        if restantes[atual.nome] > 0:
            fila.append(atual)
        else:
            concluidos += 1
    return linha

resultados_rr = {}
for quantum in (1, 2, 4):
    linha = round_robin(processos, quantum)
    resultados_rr[quantum] = linha
    gantt(linha, f'Round Robin — quantum {quantum}')
    mostrar_metricas(linha, processos)
    print()

## 4. SJF — Shortest Job First (não preemptivo)

Quando a CPU fica livre, o escalonador escolhe, entre os processos que já chegaram, aquele com menor burst. Em caso de empate, usa a ordem de chegada.

In [ ]:
def sjf(processos):
    pendentes = list(enumerate(processos))
    tempo = 0
    linha = []
    while pendentes:
        prontos = [(i, p) for i, p in pendentes if p.chegada <= tempo]
        if not prontos:
            tempo = min(p.chegada for _, p in pendentes)
            prontos = [(i, p) for i, p in pendentes if p.chegada <= tempo]
        indice, escolhido = min(prontos, key=lambda item: (item[1].burst, item[1].chegada, item[0]))
        linha.append((tempo, tempo + escolhido.burst, escolhido.nome))
        tempo += escolhido.burst
        pendentes.remove((indice, escolhido))
    return linha

linha_sjf = sjf(processos)
gantt(linha_sjf, 'SJF — não preemptivo')
mostrar_metricas(linha_sjf, processos)

## 5. Comparação de P4

Quanto menores os tempos de resposta, espera e retorno, mais rapidamente P4 é atendido e concluído.

In [ ]:
politicas = {'FCFS': linha_fcfs,
             'RR quantum 1': resultados_rr[1],
             'RR quantum 2': resultados_rr[2],
             'RR quantum 4': resultados_rr[4],
             'SJF': linha_sjf}

print('Política       | Resposta P4 | Espera P4 | Retorno P4 | Conclusão')
print('-' * 72)
for nome, linha in politicas.items():
    p4 = next(d for d in metricas(linha, processos) if d[0] == 'P4')
    fim = next(f for i, f, proc in reversed(linha) if proc == 'P4')
    print(f'{nome:<14}| {p4[1]:>11} | {p4[2]:>9} | {p4[3]:>10} | {fim:>9}')

melhor = min(politicas, key=lambda nome: next(d[3] for d in metricas(politicas[nome], processos) if d[0] == 'P4'))
print('\nMelhor política para P4 pelo menor tempo de retorno:', melhor)

## 6. Conclusão (cinco linhas)

1. No FCFS, P4 aguarda a execução dos processos que chegaram antes dele.
2. No Round Robin, o quantum altera a frequência com que P4 recebe a CPU.
3. O Round Robin com quantum 1 favoreceu P4, que terminou no tempo 6.
4. No SJF não preemptivo, P4 é escolhido após P1, pois tem o menor burst entre os prontos.
5. Assim, RR com quantum 1 foi a melhor política para P4, com conclusão no tempo 6.